Importing libraries

In [1]:
import requests
import pandas as pd
from deep_translator import GoogleTranslator

Links of URLs used for data extraction

In [2]:
urls = [
    # UNC - Nueva Caceres Discussions
    "https://www.reddit.com/r/NuevaCaceres/comments/1r8srlg/jhs/",
    "https://www.reddit.com/r/NuevaCaceres/comments/1ply212/i_need_a_friend/",
    "https://www.reddit.com/r/NuevaCaceres/comments/1nv56rs/hows_the_semester_so_far/",
    "https://www.reddit.com/r/NuevaCaceres/comments/1l6h845/bakit_sa_unc/",
    "https://www.reddit.com/r/NuevaCaceres/comments/1la8ykd/bsn_environment/",
    "https://www.reddit.com/r/NuevaCaceres/comments/1leauli/2025_the_times_higher_ranking/",
    "https://www.reddit.com/r/NuevaCaceres/comments/1lzehgk/looking_for/",
    "https://www.reddit.com/r/NuevaCaceres/comments/1m90anf/how_to_know_my_block_section/",
]

In [3]:
def translate_to_english(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except:
        return text

clean_data = []

for raw_url in urls:
    json_url = raw_url.rstrip("/") + "/.json?raw_json=1"
    headers = {"User-Agent": "unc-research-script"}
    response = requests.get(json_url, headers=headers)
    data = response.json()

    post_data = data[0]["data"]["children"][0]["data"]
    full_post_text = post_data["title"] + " " + post_data["selftext"]
    translated_post = translate_to_english(full_post_text)

    clean_data.append({
        "type": "post",
        "original_text": full_post_text,
        "translated_text": translated_post,
        "url": raw_url,
        "post_id": post_data["id"],
        "created_utc": post_data["created_utc"]
    })

    for comment in data[1]["data"]["children"]:
        if comment["kind"] == "t1":
            c = comment["data"]
            orig = c["body"]
            trans = translate_to_english(orig)

            clean_data.append({
                "type": "comment",
                "original_text": orig,
                "translated_text": trans,
                "url": raw_url,
                "post_id": post_data["id"],
                "comment_id": c["id"],
                "created_utc": c["created_utc"],
                "score": c["score"]
            })

df = pd.DataFrame(clean_data)
df.to_csv("UNCdata.csv", index=False, encoding="utf-8-sig")

print("Total rows:", len(df))

Total rows: 14
